### PASO 1: ENTRENAR YOLO PARA DETECTAR LA CUADRÍCULA

### ¿Qué hace este notebook?

Entrenamos un modelo YOLO para que, dada una foto de un sudoku,
detecte y localice la cuadrícula con una bounding box.

Este modelo es la entrada del Paso 2b: el warp necesita saber
dónde está la cuadrícula antes de corregir la perspectiva.

### Plan

```
CELDA 1 → Instalaciones e imports
CELDA 2 → Configurar rutas del dataset
CELDA 3 → Crear el archivo YAML (configuración del dataset)
CELDA 4 → Entrenar YOLO (10-15 min con GPU)
CELDA 5 → Verificar el modelo entrenado con imágenes de test
CELDA 6 → Guardar el modelo en Google Drive
```

#### CELDA 1 — Instalaciones e imports

In [ ]:
# ============================================
# INSTALACIONES E IMPORTS
# ============================================

!pip install ultralytics -q

from ultralytics import YOLO
from google.colab import drive
import cv2
import matplotlib.pyplot as plt
import shutil
import os

print("✅ Todo instalado correctamente")

#### CELDA 2 — Configurar rutas del dataset

In [ ]:
# ============================================
# RUTAS DEL DATASET
# ============================================

drive.mount('/content/drive')

# Rutas donde están las imágenes y los labels
BASE_IMAGES = '/content/drive/MyDrive/images-20260618T110202Z-3-001/images/'
BASE_LABELS = '/content/drive/MyDrive/labels-20260618T110205Z-3-001/labels/'

# Ruta destino donde guardaremos el modelo entrenado
RUTA_DESTINO_MODELO = '/content/drive/MyDrive/sudoku/modelo_yolo.pt'

# Verificar que las rutas existen antes de entrenar
print("📂 Verificando rutas del dataset:")
print(f"   Imágenes train: {os.path.exists(BASE_IMAGES + 'train/')}")
print(f"   Imágenes val:   {os.path.exists(BASE_IMAGES + 'val/')}")
print(f"   Labels train:   {os.path.exists(BASE_LABELS + 'train/')}")
print(f"   Labels val:     {os.path.exists(BASE_LABELS + 'val/')}")

# Contar imágenes disponibles
n_train = len(os.listdir(BASE_IMAGES + 'train/')) if os.path.exists(BASE_IMAGES + 'train/') else 0
n_val   = len(os.listdir(BASE_IMAGES + 'val/'))   if os.path.exists(BASE_IMAGES + 'val/')   else 0
print(f"\n📊 Imágenes disponibles:")
print(f"   Train: {n_train}")
print(f"   Val:   {n_val}")

#### CELDA 3 — Crear el archivo YAML

YOLO necesita un archivo de configuración que le diga dónde están
las imágenes y cuántas clases hay. En este caso solo hay una clase: la cuadrícula del sudoku.

In [ ]:
# ============================================
# CREAR ARCHIVO DE CONFIGURACIÓN YAML
# ============================================

yaml_content = f"""
# Dataset de Sudoku
path: /content/

train: {BASE_IMAGES}train/
val:   {BASE_IMAGES}val/
test:  {BASE_IMAGES}test/

# Solo hay 1 clase: la cuadrícula del sudoku
nc: 1
names: ['sudoku_grid']
"""

with open('/content/sudoku_data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ Archivo YAML creado: /content/sudoku_data.yaml")
print("\n📄 Contenido:")
with open('/content/sudoku_data.yaml', 'r') as f:
    print(f.read())

#### CELDA 4 — Entrenar YOLO

Entrenamos el modelo YOLO nano (el más rápido) durante 30 épocas.
Con GPU, en google colab tarda unos 10-15 minutos.

El modelo aprenderá a detectar la cuadrícula del sudoku en cualquier foto.

In [ ]:
# ============================================
# ENTRENAR YOLO
# ============================================

print("🚀 Cargando modelo YOLO preentrenado (nano)...")
modelo = YOLO('yolo11n.pt')
print(f"   Tipo: YOLO11 nano  |  Parámetros: ~2.6M")

print("\n🎯 INICIANDO ENTRENAMIENTO...")
print("="*50)
print(f"   Train: {BASE_IMAGES}train/")
print(f"   Val:   {BASE_IMAGES}val/")
print(f"   Clases: 1 (sudoku_grid)")
print(f"   Épocas: 30")
print("="*50)

resultados = modelo.train(
    data    = '/content/sudoku_data.yaml',
    epochs  = 30,      # épocas de entrenamiento
    imgsz   = 640,     # tamaño de imagen (estándar YOLO)
    batch   = 16,      # imágenes por lote
    device  = 0,       # GPU (cambiar a 'cpu' si no hay GPU)
    project = '/content/runs/detect/sudoku_yolo',
    name    = 'entrenamiento_final',
    exist_ok= True,    # sobrescribir si ya existe
    verbose = True     # mostrar progreso
)

print("\n✅ ENTRENAMIENTO COMPLETADO")
print(f"   Modelo guardado en: /content/runs/detect/sudoku_yolo/entrenamiento_final/weights/best.pt")

#### CELDA 5 — Verificar el modelo entrenado

Comprobación del modelo con las imágenes de test para confirmar que
detecta la cuadrícula correctamente antes de guardarlo.

In [ ]:
# ============================================
# VERIFICAR EL MODELO ENTRENADO
# ============================================

RUTA_MODELO = '/content/runs/detect/sudoku_yolo/entrenamiento_final/weights/best.pt'
RUTA_TEST   = '/content/drive/MyDrive/sudoku/images/test/'

# Verificar que el modelo existe
if not os.path.exists(RUTA_MODELO):
    print("❌ No se encuentra el modelo entrenado")
    print("   Ejecuta la celda 4 primero")
else:
    print("✅ Modelo encontrado")
    modelo_entrenado = YOLO(RUTA_MODELO)

    imagenes_test = [f for f in os.listdir(RUTA_TEST) if f.endswith(('.jpg', '.png'))]
    print(f"\n📊 Probando con las primeras 5 imágenes de test ({len(imagenes_test)} disponibles):")
    print("="*50)

    aciertos = 0

    for img_name in imagenes_test[:5]:
        ruta_imagen = RUTA_TEST + img_name
        results     = modelo_entrenado(ruta_imagen)

        if len(results[0].boxes) > 0:
            conf = float(results[0].boxes[0].conf[0])
            x1, y1, x2, y2 = map(int, results[0].boxes[0].xyxy[0].tolist())
            print(f"   ✅ {img_name[:40]}  →  confianza: {conf:.0%}  |  caja: ({x1},{y1})→({x2},{y2})")
            aciertos += 1

            # Mostrar imagen con la detección
            img_anotada = results[0].plot()
            plt.figure(figsize=(6, 6))
            plt.imshow(img_anotada[:, :, ::-1])   # BGR → RGB
            plt.title(f"Confianza: {conf:.0%}")
            plt.axis('off')
            plt.show()
        else:
            print(f"   ❌ {img_name[:40]}  →  no detectado")

    print(f"\n📊 Resultado: {aciertos}/5 detectados correctamente")
    if aciertos < 4:
        print("   ⚠️ Menos de 4/5 detectados → considera entrenar más épocas (epochs=50)")
    else:
        print("   ✅ Listo para guardar")

#### CELDA 6 — Guardar el modelo en Google Drive

In [ ]:
# ============================================
# GUARDAR EL MODELO EN GOOGLE DRIVE
# ============================================

RUTA_MODELO = '/content/runs/detect/sudoku_yolo/entrenamiento_final/weights/best.pt'

if not os.path.exists(RUTA_MODELO):
    print("❌ No se encuentra el modelo")
    print("   Ejecuta las celdas 4 y 5 primero")
else:
    # Crear la carpeta destino si no existe
    os.makedirs(os.path.dirname(RUTA_DESTINO_MODELO), exist_ok=True)

    shutil.copy2(RUTA_MODELO, RUTA_DESTINO_MODELO)

    if os.path.exists(RUTA_DESTINO_MODELO):
        tamaño = os.path.getsize(RUTA_DESTINO_MODELO) / (1024 * 1024)
        print(f"✅ Modelo guardado en: {RUTA_DESTINO_MODELO}")
        print(f"   Tamaño: {tamaño:.1f} MB")
        print(f"\n📌 Este modelo lo usarán el Paso 2b y el Paso 4:")
        print(f"   RUTA_YOLO = '{RUTA_DESTINO_MODELO}'")
    else:
        print("❌ No se pudo guardar el modelo")